In [ ]:
%%capture
!pip install --upgrade "kaggle-environments>=1.28.0"

In [ ]:
import math
import sys
import time
sys.path.insert(0, '/home/t/orbitwars')
from visualizer import Visualizer
from kaggle_environments import make
from kaggle_environments.envs.orbit_wars.orbit_wars import (
    Planet, Fleet, CENTER, ROTATION_RADIUS_LIMIT,
    distance, point_to_segment_distance
)

viz = Visualizer()

MAX_DISTANCE = 30
LOOK_AHEAD = 15
SHIP_SPEED_MAX = 6.0

def fleet_speed(ships):
    """Mirror the engine's speed formula exactly."""
    return min(SHIP_SPEED_MAX, 1.0 + (SHIP_SPEED_MAX - 1.0) * (math.log(ships) / math.log(1000)) ** 1.5)


def build_is_orbital(planets, initial_planets):
    """Return dict mapping Planet -> (r, initial_angle) if orbiting, else None."""
    cx = cy = CENTER
    ip_by_id = {ip[0]: ip for ip in initial_planets}
    is_orbital = {}
    for p in planets:
        r = distance((p.x, p.y), (cx, cy))
        if r + p.radius < ROTATION_RADIUS_LIMIT and p.id in ip_by_id:
            ip = ip_by_id[p.id]
            is_orbital[p] = (r, math.atan2(ip[3] - cy, ip[2] - cx))
        else:
            is_orbital[p] = None
    return is_orbital


def build_proximity_graph(planets, is_orbital, angular_velocity, scene_step, max_distance=MAX_DISTANCE, look_ahead=LOOK_AHEAD):
    """Build adjacency list: planet -> list of (neighbor, dist) within max_distance.

    scene_step is the rotation index the planets are currently at (= obs.step - 1).
    Orbiting planets are projected LOOK_AHEAD turns into the future.
    """
    cx = cy = CENTER

    future_pos = {}
    for p in planets:
        orb = is_orbital[p]
        if orb is not None:
            r, ia = orb
            a = ia + angular_velocity * (scene_step + look_ahead)
            future_pos[p] = (cx + r * math.cos(a), cy + r * math.sin(a))
        else:
            future_pos[p] = (p.x, p.y)

    graph = {p: [] for p in planets}
    for i, a in enumerate(planets):
        ax, ay = future_pos[a]
        for b in planets[i + 1:]:
            bx, by = future_pos[b]
            dist = distance((ax, ay), (bx, by))
            if dist <= max_distance:
                graph[a].append((b, dist))
                graph[b].append((a, dist))
    return graph, future_pos


def viz_proximity_graph(viz, scene_step, planets, graph, future_pos):
    """Draw graph edges and future-position planet labels onto the visualizer frame."""
    moving = {p for p in planets if future_pos[p] != (p.x, p.y)}
    seen_edges = set()
    for p, neighbors in graph.items():
        fpx, fpy = future_pos[p]
        if p in moving:
            viz.add_label(scene_step, fpx, fpy, f'P{p.id}', color='#22ffcc')
        for nb, dist in neighbors:
            edge = (min(p.id, nb.id), max(p.id, nb.id))
            if edge in seen_edges:
                continue
            seen_edges.add(edge)
            nx, ny = future_pos[nb]
            viz.add_line(scene_step, fpx, fpy, nx, ny, color='#22aaff', width=1)


def intercept_angle(sx, sy, target, is_orbital, angular_velocity, scene_step, ships, tol=1e-6, max_iters=30):
    """Aim angle from (sx, sy) toward where target will be when a fleet arrives.

    scene_step is the rotation index planets are currently at (= obs.step - 1).

    Fleet launched this turn moves for T steps. The engine checks collision on
    movement step T against the planet at rotation scene_step + (T-1), because
    planets rotate AFTER each fleet-movement step. So we target the planet at
    abs_step = scene_step + travel - 1.

    For non-orbital planets the position is constant, so no iteration needed.
    For orbital planets, uses damped fixed-point iteration to avoid oscillation
    when angular velocity is high. Converges when travel changes by < tol steps.
    Returns (angle, intercept_x, intercept_y, travel_steps).
    """
    speed = fleet_speed(ships)
    orb = is_orbital[target]
    if orb is None:
        tx, ty = target.x, target.y
        travel = distance((sx, sy), (tx, ty)) / speed
    else:
        cx = cy = CENTER
        r, ia = orb
        # Seed: straight-line travel time to the planet's current position.
        travel = distance((sx, sy), (target.x, target.y)) / speed
        tx, ty = target.x, target.y
        for _ in range(max_iters):
            a = ia + angular_velocity * (scene_step + travel - 1)
            new_tx, new_ty = cx + r * math.cos(a), cy + r * math.sin(a)
            new_travel = distance((sx, sy), (new_tx, new_ty)) / speed
            # Damp update: average old and new travel to suppress oscillation.
            new_travel = 0.5 * (travel + new_travel)
            if abs(new_travel - travel) < tol:
                tx, ty = new_tx, new_ty
                travel = new_travel
                break
            travel = new_travel
            tx, ty = new_tx, new_ty
    angle = math.atan2(ty - sy, tx - sx)
    return angle, tx, ty, travel


def build_destination_list(fleets, planets, is_orbital, angular_velocity, scene_step):
    """For each fleet, find the first planet it is on an interception course for.

    scene_step is the rotation index planets are currently at (= obs.step - 1).

    For each planet, computes the intercept angle the fleet would need to hit it,
    then checks whether the fleet's actual heading is within the hit cone
    (half-angle = asin(planet.radius / dist_to_intercept)). Returns the earliest-
    arriving planet the fleet is actually aimed at.

    Returns: dict mapping Fleet -> (Planet, t, arrival_x, arrival_y).
             t is continuous time in turns.
    """
    destination_list = {}
    for fleet in fleets:
        best = None
        best_t = float('inf')
        for planet in planets:
            needed_angle, px, py, travel = intercept_angle(
                fleet.x, fleet.y, planet, is_orbital, angular_velocity, scene_step, fleet.ships
            )
            dist = distance((fleet.x, fleet.y), (px, py))
            if dist < planet.radius:
                half_cone = math.pi
            else:
                half_cone = math.asin(min(1.0, planet.radius / dist))
            delta = abs(math.atan2(math.sin(fleet.angle - needed_angle),
                                   math.cos(fleet.angle - needed_angle)))
            if delta <= half_cone and travel < best_t:
                best_t = travel
                best = (planet, travel, px, py)
        if best is not None:
            destination_list[fleet] = best
    return destination_list


def viz_destination_list(viz, scene_step, destination_list):
    """Draw a line from each fleet to its destination planet's arrival position."""
    lines = []
    for fleet, (planet, t, px, py) in destination_list.items():
        color = '#44ff88' if fleet.owner == 0 else '#ff8844'
        viz.add_line(scene_step, fleet.x, fleet.y, px, py, color=color, width=1)
        lines.append(f' {fleet.owner} F{fleet.id} P{fleet.from_planet_id}({fleet.ships}) -> P{planet.id}({planet.ships}) t={round(t)} v={fleet_speed(fleet.ships):.2f}')
    if lines:
        viz.add_text(scene_step, 'dest_list:\n' + '\n'.join(lines))


def exec_snipe(viz, scene_step, owned_planets, enemy_planets, moves, is_orbital, angular_velocity):
    best_travel = float('inf')
    best_mine = None
    best_enemy = None
    for mine in owned_planets:
        for enemy in enemy_planets:
            _, _, _, travel = intercept_angle(mine.x, mine.y, enemy, is_orbital, angular_velocity, scene_step, mine.ships)
            if travel < best_travel:
                best_travel = travel
                best_mine = mine
                best_enemy = enemy
    if best_mine is None or best_mine.ships < 5:
        reason = 'no owned planet' if best_mine is None else f'P{best_mine.id} ships={best_mine.ships} < 5'
        viz.add_text(scene_step, f'snipe: skip ({reason})')
        return
    ships = best_mine.ships
    angle, ix, iy, travel = intercept_angle(best_mine.x, best_mine.y, best_enemy, is_orbital, angular_velocity, scene_step, ships=ships)
    moves.append([best_mine.id, angle, ships])
    viz.add_text(scene_step, f'snipe: P{best_mine.id}({ships}sh) -> P{best_enemy.id}({best_enemy.ships}sh) angle={angle:.3f} travel={travel:.1f}t')
    viz.add_line(scene_step, best_mine.x, best_mine.y, ix, iy, color='#ff44ff', width=2)


def hellburner(obs):
    viz.record(obs)
    _t0 = time.perf_counter()

    moves = []
    player = obs['player']

    # obs.step=N means the engine has completed N-1 rotations, so planets sit at N-1.
    scene_step = obs['step'] - 1

    angular_velocity = obs.get('angular_velocity', 0.0)
    planets = [Planet(*p) for p in obs['planets']]
    owned_planets = [p for p in planets if p.owner == player]
    comet_ids = set(obs.get('comet_planet_ids', []))
    enemy_planets = [p for p in planets if p.owner != player and p.id not in comet_ids]

    if not enemy_planets:
        return moves

    fleets = [Fleet(*f) for f in obs['fleets']]
    owned_fleets = [f for f in fleets if f.owner == player]
    enemy_fleets = [f for f in fleets if f.owner != player]

    is_orbital = build_is_orbital(planets, obs.get('initial_planets', []))

    graph, future_pos = build_proximity_graph(planets, is_orbital, angular_velocity, scene_step)

    destination_list = build_destination_list(fleets, planets, is_orbital, angular_velocity, scene_step)

    elapsed_ms = (time.perf_counter() - _t0) * 1000
    viz.add_text(scene_step, f'hellburner ms: {elapsed_ms:.2f}ms')
    #viz_proximity_graph(viz, scene_step, planets, graph, future_pos)
    viz_destination_list(viz, scene_step, destination_list)

    exec_snipe(viz, scene_step, owned_planets, enemy_planets, moves, is_orbital, angular_velocity)

    return moves


In [ ]:
# Run game and save visualizer
env = make('orbit_wars', debug=False)
env.run([hellburner, lambda obs: []])

final = env.steps[-1]
for i, s in enumerate(final):
    print(f'Player {i}: reward={s.reward}, status={s.status}')

#env.render(mode='ipython', width=800, height=600)
viz.save('/mnt/c/Users/ajohn/Downloads/orbitwars_viz.html')